# Phase 4 — Does the null hold at a longer horizon?

Budget: ~5 GPU-hr, one session
Config: 1 seed (42), 3000 steps (≈2.2 hr baseline / 2.7 hr QGFD)

Key changes from Phase 2:
- save_strategy='steps', save_steps=500 (so session timeout doesn't lose the run — resume via resume_from_checkpoint)
- eval_strategy='steps', eval_steps=200 to get an eval-loss TRAJECTORY, not just an endpoint
- Plot both eval loss curves over training to see if QGFD separates from baseline later in training

## 0. Install & Setup

In [ ]:
import os
# Pin single GPU to avoid DataParallel / bitsandbytes crashes
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install -q git+https://github.com/rajboopathiking/TorchDire.git
!pip install -q transformers datasets peft trl bitsandbytes accelerate matplotlib pandas

In [ ]:
import torch
import gc
import json
import glob
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig, set_seed
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from torchdire import patch_llama_with_qgfd, register_qgfd_step_callback

## 1. Configuration

In [ ]:
MODEL_ID = "42dot/42dot_LLM-SFT-1.3B"
DATASET_ID = "arbml/alpagasus_cleaned"
SEED = 42
MAX_STEPS = 3000
EVAL_STEPS = 200
SAVE_STEPS = 500
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
QGFD_ALPHA = 0.10  # Best alpha from Phase 3
OUTPUT_DIR = "/kaggle/working"

## 2. Helper Functions

In [ ]:
def cleanup_memory():
    gc.collect()
    torch.cuda.empty_cache()

def format_example(example):
    return f"Instruction: {example['instruction']}\nInput: {example.get('input', '')}\nOutput: {example['output']}"

def make_sft_config(run_name):
    out_dir = os.path.join(OUTPUT_DIR, run_name)
    return TrainingArguments(
        output_dir=out_dir,
        max_steps=MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        bf16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        report_to="none",
        gradient_checkpointing=True,
        seed=SEED,
        optim="paged_adamw_32bit"
    )

def run_long_arm(is_baseline=True):
    set_seed(SEED)
    cleanup_memory()
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    
    dataset = load_dataset(DATASET_ID)
    split_dataset = dataset["train"].train_test_split(test_size=0.1, seed=SEED)
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model.config.use_cache = False
    
    model = prepare_model_for_kbit_training(model)
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    
    if not is_baseline:
        patch_llama_with_qgfd(model, alpha=QGFD_ALPHA)
        
    run_name = "baseline" if is_baseline else f"qgfd_alpha{QGFD_ALPHA}"
    out_dir = os.path.join(OUTPUT_DIR, run_name)
    
    args = make_sft_config(run_name)
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=split_dataset["train"],
        eval_dataset=split_dataset["test"],
        args=args,
        peft_config=lora_config,
    )
    
    if not is_baseline:
        register_qgfd_step_callback(trainer, model)
        
    # Check for existing checkpoints to resume
    checkpoints = glob.glob(os.path.join(out_dir, "checkpoint-*"))
    latest_checkpoint = None
    if len(checkpoints) > 0:
        latest_checkpoint = max(checkpoints, key=os.path.getctime)
        print(f"Resuming from checkpoint: {latest_checkpoint}")
        
    trainer.train(resume_from_checkpoint=latest_checkpoint)
    
    # Save log history
    log_file = os.path.join(OUTPUT_DIR, f"{run_name}_log_history.json")
    with open(log_file, "w") as f:
        json.dump(trainer.state.log_history, f)
        
    # Extract eval loss trajectory
    eval_logs = [log for log in trainer.state.log_history if 'eval_loss' in log]
    df = pd.DataFrame(eval_logs)
    csv_file = os.path.join(OUTPUT_DIR, f"phase4_{run_name}_eval_trajectory.csv")
    df.to_csv(csv_file, index=False)
    print(f"Saved trajectory to {csv_file}")
    
    del model
    del trainer
    cleanup_memory()
    
    return df

## 3. Run Baseline Arm

In [ ]:
baseline_df = run_long_arm(is_baseline=True)
baseline_df.head()

## 4. Run QGFD Arm

In [ ]:
qgfd_df = run_long_arm(is_baseline=False)
qgfd_df.head()

## 5. Extract Eval Loss Trajectories & 6. Plot Learning Curves

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(baseline_df['step'], baseline_df['eval_loss'], label='Baseline', marker='o')
plt.plot(qgfd_df['step'], qgfd_df['eval_loss'], label=f'QGFD (alpha={QGFD_ALPHA})', marker='x')
plt.xlabel('Training Steps')
plt.ylabel('Evaluation Loss')
plt.title('Evaluation Loss Trajectory: Baseline vs QGFD (3000 steps)')
plt.legend()
plt.grid(True)

plot_file = os.path.join(OUTPUT_DIR, "phase4_eval_loss_trajectory.png")
plt.savefig(plot_file)
plt.show()
print(f"Plot saved to {plot_file}")

## 7. Endpoint Comparison & 8. Analysis

In [ ]:
final_baseline_loss = baseline_df.iloc[-1]['eval_loss']
final_qgfd_loss = qgfd_df.iloc[-1]['eval_loss']

print("=== Endpoint Comparison ===")
print(f"Final Baseline Eval Loss: {final_baseline_loss:.4f}")
print(f"Final QGFD Eval Loss:     {final_qgfd_loss:.4f}")
print(f"Difference:               {final_baseline_loss - final_qgfd_loss:.4f}")

import math
print("\n=== Perplexity (PPL) ===")
print(f"Baseline PPL: {math.exp(final_baseline_loss):.4f}")
print(f"QGFD PPL:     {math.exp(final_qgfd_loss):.4f}")

# Output final metrics to CSV
summary_df = pd.DataFrame({
    'Run': ['Baseline', 'QGFD'],
    'Final_Eval_Loss': [final_baseline_loss, final_qgfd_loss],
    'Final_PPL': [math.exp(final_baseline_loss), math.exp(final_qgfd_loss)]
})
summary_csv = os.path.join(OUTPUT_DIR, "phase4_endpoint_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved endpoint summary to {summary_csv}")

## 9. Final Summary

*Does QGFD separate from baseline at any point?*

*(Analysis to be filled based on output...)*